# Viraltest v2 — Real LLM Training with LoRA + Environment Rewards

This notebook **actually trains** an LLM (Qwen2.5-1.5B-Instruct) to play our Instagram creator simulation.

**Pipeline:**
1. Clone repo & install deps
2. Run 5 heuristic baselines × 3 tasks (15 runs) → leaderboard
3. Run **untrained** LLM on all 3 tasks → "before" scores
4. **LoRA fine-tune** with reward-weighted SFT (4 rounds × 6 episodes = real weight updates)
5. Run **trained** LLM on all 3 tasks → "after" scores
6. Generate real plots from real numbers

**Requirements:** Colab T4 GPU (free tier), ~45 min total.

**What makes this real training:** LoRA adapter weights are actually updated via gradient descent. The model's behavior changes because its weights change, not because we edit the prompt.

In [5]:
# Cell 1: Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q transformers>=4.45.0 accelerate peft>=0.10.0 trl>=0.20.0 datasets bitsandbytes
!pip install -q matplotlib pandas
!pip install -q pydantic httpx
!pip install -q "openenv-core[core]>=0.2.2"


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
zsh:1: 4.45.0 not found

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
# Cell 2: Resolve repo path (Colab: fresh clone. Local: auto-detect project root)
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_BRANCH = "hack1"
REPO_URL = "https://github.com/VaibhavKhandare/viral-posts-env.git"
COLAB_REPO = Path("/content/viral-posts-env")


def _is_repo_root(p: Path) -> bool:
    return (p / "server" / "viraltest_environment.py").is_file() and (p / "models.py").is_file()


def _find_local_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, here.parent, here.parent.parent):
        if _is_repo_root(cand):
            return cand
    raise FileNotFoundError(
        "Could not find project root. cd into viral-posts-env or run this notebook in Google Colab."
    )


# --- Colab: always clone a clean copy (avoids stale 7-day code) ---
if Path("/content").is_dir():
    if COLAB_REPO.exists():
        shutil.rmtree(COLAB_REPO, ignore_errors=True)
    p = subprocess.run(
        [
            "git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
            REPO_URL, str(COLAB_REPO),
        ],
        capture_output=True,
        text=True,
    )
    if p.returncode != 0:
        raise RuntimeError(
            "git clone failed. Check network and branch name.\n"
            f"stdout:\n{p.stdout}\nstderr:\n{p.stderr}"
        )
    if not COLAB_REPO.is_dir():
        raise FileNotFoundError(f"Clone did not create {COLAB_REPO}")
    os.chdir(COLAB_REPO)
    print("Mode: Colab (fresh clone)")
else:
    # --- Local machine: do not use /content ---
    root = _find_local_root()
    os.chdir(root)
    print("Mode: local")
    print(f"Repo root: {root}")

REPO_DIR = str(Path.cwd().resolve())
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

PLOTS_DIR = os.path.join(REPO_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

try:
    commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        stderr=subprocess.DEVNULL,
        text=True,
    ).strip()
except Exception:
    commit = "n/a"

print(f"Working dir: {os.getcwd()}")
print(f"Branch: {REPO_BRANCH}")
print(f"Commit: {commit}")
print(f"Plots dir: {PLOTS_DIR}")

fatal: could not create leading directories of '/content/viral-posts-env': Read-only file system


FileNotFoundError: [Errno 2] No such file or directory: '/content/viral-posts-env'

In [ ]:
# Cell 3: Imports (with runtime validation)
import json, random, time, textwrap, copy, os, sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict

# Find repo root if notebook was opened from training/ and Cell 2 was skipped
if not Path("server/viraltest_environment.py").is_file():
    for cand in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (cand / "server" / "viraltest_environment.py").is_file():
            os.chdir(cand)
            s = str(cand.resolve())
            if s not in sys.path:
                sys.path.insert(0, s)
            print("Auto chdir to repo root:", s)
            break
    else:
        raise RuntimeError(
            "Project files not found. Run **Cell 2** first (Colab), or run from repo root.\n"
            f"  cwd = {os.getcwd()!r}\n"
        )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from models import ScheduledAction, ToolCall, ViraltestAction
from server.viraltest_environment import (
    ViraltestEnvironment, TAG_POOL, TASK_HORIZON,
    TOPIC_CATEGORIES,
)

ALL_TOPICS = [t for topics in TOPIC_CATEGORIES.values() for t in topics]
NICHES = list(TOPIC_CATEGORIES.keys())
CONTENT_TYPES = ["reel", "carousel", "story", "text_post"]
INTENTS = ["send_bait", "save_bait", "watch_bait", "like_bait"]
TASKS = ["monthly_engage", "monthly_strategic", "monthly_competitive"]

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Tags: {len(TAG_POOL)}, Topics: {len(ALL_TOPICS)}, Horizon: {TASK_HORIZON} days")

# Hard stop if stale repo/code is loaded
assert TASK_HORIZON == 30, (
    f"Expected TASK_HORIZON=30, got {TASK_HORIZON}. "
    "Restart runtime and run from Cell 1 again (clean clone on hack1)."
)

## Part 1: Heuristic Baselines

5 scripted agents prove the environment differentiates skill levels.

In [ ]:
# Cell 4: Define heuristic agents + episode runner
_rng = random.Random(42)

def plan_always_rest(obs_dict, day):
    return ViraltestAction(scheduled_actions=[])

def plan_spam(obs_dict, day):
    return ViraltestAction(scheduled_actions=[
        ScheduledAction(hour=h, action_type="post", content_type="reel",
                        topic="AI tools", tags=["ai"], intent="watch_bait")
        for h in range(24)])

def plan_random(obs_dict, day):
    actions = []
    for h in range(24):
        if _rng.random() < 0.1:
            actions.append(ScheduledAction(
                hour=h, action_type="post",
                content_type=_rng.choice(CONTENT_TYPES),
                topic=_rng.choice(ALL_TOPICS),
                tags=_rng.sample(TAG_POOL[:30], 3),
                intent=_rng.choice(INTENTS)))
    return ViraltestAction(scheduled_actions=actions)

def plan_minimal(obs_dict, day):
    return ViraltestAction(scheduled_actions=[
        ScheduledAction(hour=12, action_type="post", content_type="carousel",
                        topic=ALL_TOPICS[day % len(ALL_TOPICS)],
                        tags=[TAG_POOL[i % len(TAG_POOL)] for i in range(day, day+3)],
                        intent="save_bait")])

def plan_smart(obs_dict, day):
    return ViraltestAction(
        tool_calls=[ToolCall(name="query_trends",
                   arguments={"niche": NICHES[day % len(NICHES)]})] if day <= 3 else [],
        scheduled_actions=[
            ScheduledAction(hour=8, action_type="create_content"),
            ScheduledAction(hour=12, action_type="post",
                content_type=CONTENT_TYPES[(day*2)%4],
                topic=ALL_TOPICS[(day*2)%len(ALL_TOPICS)],
                tags=[TAG_POOL[(day*6+i)%len(TAG_POOL)] for i in range(3)],
                intent=INTENTS[(day*2)%4]),
            ScheduledAction(hour=19, action_type="post",
                content_type=CONTENT_TYPES[(day*2+1)%4],
                topic=ALL_TOPICS[(day*2+1)%len(ALL_TOPICS)],
                tags=[TAG_POOL[(day*6+3+i)%len(TAG_POOL)] for i in range(3)],
                intent=INTENTS[(day*2+1)%4]),
        ],
        replies=[{"post_hour": 12, "reply_hour": 13}])

BASELINE_AGENTS = {
    "always_rest": plan_always_rest, "spam": plan_spam,
    "random": plan_random, "minimal": plan_minimal, "smart": plan_smart,
}

def run_episode(task, plan_fn, seed=42):
    env = ViraltestEnvironment()
    obs = env.reset(task=task, seed=seed)
    obs_dict = obs.model_dump()
    rewards, energies = [], [obs.creator_energy]
    for day in range(1, TASK_HORIZON + 1):
        action = plan_fn(obs_dict, day)
        obs = env.step(action)
        obs_dict = obs.model_dump()
        rewards.append(obs.reward or 0.0)
        energies.append(obs.creator_energy)
        if obs.done: break
    grader = (obs.metadata or {}).get("grader_score", 0.0)
    return {"grader_score": grader, "total_reward": sum(rewards),
            "steps": len(rewards), "final_energy": obs.creator_energy,
            "follower_delta": obs.follower_count - 10000,
            "burned_out": obs.creator_energy <= 0,
            "rewards": rewards, "energies": energies}

print("Agents and episode runner defined.")

In [ ]:
# Cell 5: Run baselines (safe)
print("Running heuristic baselines (5 agents × 3 tasks)...")
print("=" * 70)

required = ["BASELINE_AGENTS", "run_episode", "TASKS", "random"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(
        f"Missing prerequisites: {missing}. Run notebook from top (Cell 1 -> Cell 5)."
    )

baseline_results = {}
for name, fn in BASELINE_AGENTS.items():
    baseline_results[name] = {}
    for task in TASKS:
        _rng = random.Random(42)
        try:
            result = run_episode(task, fn, seed=42)
        except Exception as e:
            raise RuntimeError(
                f"Baseline failed for agent={name}, task={task}: {type(e).__name__}: {e}"
            ) from e
        baseline_results[name][task] = result
        print(f"  {name:>12s} | {task:>22s} | score={result['grader_score']:.4f} "
              f"| energy={result['final_energy']:.2f}")
    print()

print("\nLEADERBOARD")
print(f"{'Agent':<14s} {'Engage':>10s} {'Strategic':>12s} {'Competitive':>14s} {'Avg':>8s}")
print("-" * 60)
for name in BASELINE_AGENTS:
    scores = [baseline_results[name][t]["grader_score"] for t in TASKS]
    print(f"{name:<14s} {scores[0]:>10.4f} {scores[1]:>12.4f} {scores[2]:>14.4f} {sum(scores)/3:>8.4f}")

In [ ]:
# Cell 6: Baseline plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
agent_names = list(BASELINE_AGENTS.keys())
colors = ['#E53935', '#FF9800', '#9E9E9E', '#42A5F5', '#4CAF50']
for i, task in enumerate(TASKS):
    scores = [baseline_results[a][task]["grader_score"] for a in agent_names]
    bars = axes[i].barh(agent_names, scores, color=colors)
    axes[i].set_title(task.replace("monthly_", "").title(), fontsize=13, fontweight='bold')
    for bar, score in zip(bars, scores):
        axes[i].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                     f"{score:.4f}", va='center', fontsize=9)
axes[0].set_ylabel("Agent")
fig.suptitle("Viraltest v2 — Heuristic Baseline Leaderboard", fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/baseline_leaderboard.png", dpi=150, bbox_inches='tight')
plt.show()

## Part 2: Load LLM (Qwen2.5-1.5B-Instruct)

We load the base model with 4-bit quantization to fit in free Colab's T4 GPU (16GB VRAM).

In [ ]:
# Cell 7: Load model
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} (4-bit quantized)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print(f"Model loaded. Device: {model.device}")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Cell 8: LLM agent functions
SYSTEM_PROMPT = textwrap.dedent("""\
You are an Instagram content strategy agent. Each step is one day.
You manage a creator account over a 30-day cycle.

RESPONSE FORMAT — return ONLY valid JSON, no markdown:
{
  "tool_calls": [{"name": "query_trends", "arguments": {"niche": "tech"}}],
  "scheduled_actions": [
    {"hour": 12, "action_type": "post", "content_type": "reel",
     "topic": "AI tools", "tags": ["ai", "coding"], "intent": "watch_bait"}
  ],
  "replies": [{"post_hour": 12, "reply_hour": 13}],
  "notes": "strategy notes"
}

RULES:
- content_type: reel|story|carousel|text_post
- intent: send_bait|save_bait|watch_bait|like_bait
- 1-2 posts/day optimal. More = fatigue.
- Empty scheduled_actions = rest (recovers energy).
- Vary content types and topics for diversity bonus.
- Reply within 90 min of post for reach bonus.""")


def format_obs(obs):
    days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    day_name = days[obs.day_of_week] if 0 <= obs.day_of_week < 7 else "?"
    signals_str = ""
    signals = getattr(obs, "engagement_signals", None)
    if signals:
        signals_str = (f"Signals: watch={signals.watch_time:.3f} "
                       f"sends={signals.sends_per_reach:.3f} "
                       f"saves={signals.saves:.3f}\n")
    tool_str = ""
    for tr in getattr(obs, "tool_results", []):
        if tr.success:
            tool_str += f"  {tr.name}: {json.dumps(tr.data)[:200]}\n"
    return (f"Day: {day_name} | days_elapsed={obs.days_elapsed}\n"
            f"Energy: {obs.creator_energy:.2f} | Followers: {obs.follower_count}\n"
            f"Engagement: {obs.engagement_rate:.3f} | Queue: {obs.content_queue_size}\n"
            f"{signals_str}"
            f"Tool results:\n{tool_str if tool_str else '  (none)\n'}"
            f"Plan your actions (JSON only):")


def parse_model_output(text):
    text = text.strip()
    if "```" in text:
        lines = [l for l in text.split("\n") if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()
    start, end = text.find("{"), text.rfind("}") + 1
    if start >= 0 and end > start:
        text = text[start:end]
    try:
        data = json.loads(text)
        tool_calls = [ToolCall(name=tc["name"], arguments=tc.get("arguments", {}))
                      for tc in data.get("tool_calls", []) if isinstance(tc, dict) and "name" in tc]
        scheduled = []
        for a in data.get("scheduled_actions", []):
            try: scheduled.append(ScheduledAction(**a))
            except: pass
        return ViraltestAction(tool_calls=tool_calls, scheduled_actions=scheduled,
                               replies=data.get("replies", []), notes=data.get("notes"))
    except:
        return ViraltestAction(scheduled_actions=[])


def generate_action(mdl, tok, obs, history, temperature=0.7):
    prompt = format_obs(obs)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history[-4:])
    messages.append({"role": "user", "content": prompt})
    text_input = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text_input, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=512, temperature=temperature,
                           do_sample=True, top_p=0.9, pad_token_id=tok.eos_token_id)
    resp = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return resp, parse_model_output(resp)


def run_llm_episode(mdl, tok, task, seed=42, verbose=False):
    env = ViraltestEnvironment()
    obs = env.reset(task=task, seed=seed)
    rewards, energies = [], [obs.creator_energy]
    history, pairs = [], []
    for day in range(1, TASK_HORIZON + 1):
        if obs.done: break
        if obs.creator_energy <= 0.25:
            action = ViraltestAction(scheduled_actions=[])
            resp = '{"scheduled_actions": []}'
        else:
            resp, action = generate_action(mdl, tok, obs, history)
        prompt = format_obs(obs)
        pairs.append({"prompt": prompt, "response": resp})
        obs = env.step(action)
        r = obs.reward or 0.0
        rewards.append(r)
        energies.append(obs.creator_energy)
        history.extend([{"role": "user", "content": prompt},
                        {"role": "assistant", "content": resp}])
        if verbose:
            n_p = len([s for s in action.scheduled_actions if s.action_type=="post"])
            print(f"    Day {day:2d}: r={r:.4f} e={obs.creator_energy:.2f} posts={n_p} tools={len(action.tool_calls)}")
        if obs.done: break
    gs = (obs.metadata or {}).get("grader_score", 0.0)
    return {"task": task, "grader_score": gs, "total_reward": sum(rewards),
            "final_energy": obs.creator_energy, "rewards": rewards,
            "energies": energies, "pairs": pairs,
            "follower_delta": obs.follower_count - 10000,
            "burned_out": obs.creator_energy <= 0}

print("LLM agent functions defined.")

## Part 3: Untrained LLM Baseline (“Before”)

Run the base model with NO fine-tuning. This establishes ground truth.

In [ ]:
# Cell 9: Run untrained model
print("Running UNTRAINED base model on all tasks...")
print("=" * 60)

before_results = {}
for task in TASKS:
    print(f"\n  Task: {task}")
    result = run_llm_episode(model, tokenizer, task, seed=42, verbose=True)
    before_results[task] = result
    print(f"  => grader={result['grader_score']:.4f} reward={result['total_reward']:.3f}")

print("\n" + "=" * 60)
print("BEFORE TRAINING:")
for t in TASKS:
    print(f"  {t}: grader={before_results[t]['grader_score']:.4f}")

## Part 4: LoRA Fine-Tuning (Real Weight Updates)

This is the core training loop. For each round:
1. Collect episodes with current model
2. Score each (prompt, response) pair by episode reward
3. Keep top 50% highest-reward samples
4. Fine-tune LoRA weights via SFT on those samples

The model's actual weights change via gradient descent — this is real training.

In [ ]:
# Cell 10: Attach LoRA adapter
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM, bias="none",
)

model.enable_input_require_grads()
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
# Cell 11: Training loop
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

NUM_ROUNDS = 4
EPISODES_PER_ROUND = 6
TOP_K_FRACTION = 0.5

training_log = {
    "round": [], "avg_episode_reward": [], "max_episode_reward": [],
    "min_episode_reward": [], "avg_grader": [], "max_grader": [],
    "n_training_samples": [], "train_loss": [],
}

t_start = time.time()

for round_idx in range(1, NUM_ROUNDS + 1):
    print(f"\n{'=' * 60}")
    print(f"TRAINING ROUND {round_idx}/{NUM_ROUNDS}")
    print(f"{'=' * 60}")

    # Collect episodes
    peft_model.eval()
    all_pairs, episode_rewards, episode_graders = [], [], []

    for ep in range(EPISODES_PER_ROUND):
        task = TASKS[ep % len(TASKS)]
        seed = 42 + (round_idx - 1) * 100 + ep
        result = run_llm_episode(peft_model, tokenizer, task, seed=seed)
        ep_reward = result["total_reward"] + 2.0 * result["grader_score"]
        episode_rewards.append(ep_reward)
        episode_graders.append(result["grader_score"])

        for pr in result["pairs"]:
            text = (f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                    f"<|im_start|>user\n{pr['prompt']}<|im_end|>\n"
                    f"<|im_start|>assistant\n{pr['response']}<|im_end|>")
            all_pairs.append({"text": text, "reward": ep_reward})

        print(f"  ep {ep+1}/{EPISODES_PER_ROUND}: {task.split('_')[-1]:>11s} "
              f"grader={result['grader_score']:.4f} reward={ep_reward:.3f}")

    avg_r = np.mean(episode_rewards)
    avg_g = np.mean(episode_graders)
    print(f"  Avg reward={avg_r:.3f} Avg grader={avg_g:.4f}")

    # Filter to top-K
    threshold = np.percentile([p["reward"] for p in all_pairs], (1 - TOP_K_FRACTION) * 100)
    filtered = [p for p in all_pairs if p["reward"] >= threshold] or all_pairs
    print(f"  Filtered to {len(filtered)}/{len(all_pairs)} samples")

    dataset = Dataset.from_list([{"text": p["text"]} for p in filtered])

    # SFT training (real gradient updates)
    sft_config = SFTConfig(
        output_dir=f"./checkpoints/round_{round_idx}",
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        warmup_steps=5,
        logging_steps=5,
        save_strategy="no",
        max_length=1024,
        fp16=True,
        report_to="none",
    )

    peft_model.train()
    trainer = SFTTrainer(
        model=peft_model, processing_class=tokenizer,
        train_dataset=dataset, args=sft_config,
    )
    train_result = trainer.train()
    loss = train_result.training_loss
    print(f"  Training loss: {loss:.4f}")

    training_log["round"].append(round_idx)
    training_log["avg_episode_reward"].append(round(float(avg_r), 3))
    training_log["max_episode_reward"].append(round(float(max(episode_rewards)), 3))
    training_log["min_episode_reward"].append(round(float(min(episode_rewards)), 3))
    training_log["avg_grader"].append(round(float(avg_g), 4))
    training_log["max_grader"].append(round(float(max(episode_graders)), 4))
    training_log["n_training_samples"].append(len(filtered))
    training_log["train_loss"].append(round(loss, 4))

elapsed = time.time() - t_start
print(f"\nTraining complete in {elapsed/60:.1f} min")
print(pd.DataFrame(training_log).to_string(index=False))

## Part 5: Trained LLM Evaluation (“After”)

Same model, same seeds, same environment — but now with updated LoRA weights.

In [ ]:
# Cell 12: Run trained model
print("Running TRAINED model on all tasks...")
print("=" * 60)

peft_model.eval()
after_results = {}
for task in TASKS:
    print(f"\n  Task: {task}")
    result = run_llm_episode(peft_model, tokenizer, task, seed=42, verbose=True)
    after_results[task] = result
    print(f"  => grader={result['grader_score']:.4f} reward={result['total_reward']:.3f}")

print("\n" + "=" * 60)
print("AFTER TRAINING:")
for t in TASKS:
    print(f"  {t}: grader={after_results[t]['grader_score']:.4f}")

## Part 6: Result Plots — Real Training Evidence

In [ ]:
# Cell 13: Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rounds = training_log["round"]

axes[0].plot(rounds, training_log["avg_grader"], 'o-', color='#2196F3', lw=2, label='Avg grader')
axes[0].fill_between(rounds, training_log["avg_grader"],
                     training_log["max_grader"], alpha=0.2, color='#2196F3')
axes[0].set_xlabel('Round'); axes[0].set_ylabel('Grader Score')
axes[0].set_title('Grader Score Over Rounds', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(rounds, training_log["train_loss"], 's-', color='#E53935', lw=2)
axes[1].set_xlabel('Round'); axes[1].set_ylabel('Loss')
axes[1].set_title('Training Loss', fontweight='bold')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Viraltest v2 — LoRA Training Progress (Qwen 1.5B)', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{PLOTS_DIR}/reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 14: Before vs After
task_labels = [t.replace('monthly_', '').title() for t in TASKS]
x = np.arange(len(TASKS))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
b_scores = [before_results[t]["grader_score"] for t in TASKS]
a_scores = [after_results[t]["grader_score"] for t in TASKS]
s_scores = [baseline_results["smart"][t]["grader_score"] for t in TASKS]

ax.bar(x - w, b_scores, w, label='Base Model (Before)', color='#FF9800')
ax.bar(x, a_scores, w, label='LoRA Trained (After)', color='#4CAF50')
ax.bar(x + w, s_scores, w, label='Smart Heuristic', color='#9E9E9E', alpha=0.7)

ax.set_ylabel('Grader Score'); ax.set_xticks(x); ax.set_xticklabels(task_labels)
ax.set_title('Before vs After LoRA Training — Grader Scores', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

for container in ax.containers:
    for bar in container:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2., h + 0.005,
                    f'{h:.4f}', ha='center', va='bottom', fontsize=9)

fig.tight_layout()
fig.savefig(f'{PLOTS_DIR}/before_after.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 15: Trajectory comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
comparisons = [
    ("Base Model", before_results, '#FF9800', '--'),
    ("LoRA Trained", after_results, '#4CAF50', '-'),
]
for i, task in enumerate(TASKS):
    for label, res, color, ls in comparisons:
        lw = 2.5 if 'Trained' in label else 1.5
        axes[0, i].plot(res[task]["rewards"], label=label, color=color, lw=lw, ls=ls)
        axes[1, i].plot(res[task]["energies"], label=label, color=color, lw=lw, ls=ls)
    sr = baseline_results["smart"][task]
    axes[0, i].plot(sr["rewards"], label="Smart", color='#9E9E9E', lw=1, ls=':')
    axes[1, i].plot(sr["energies"], label="Smart", color='#9E9E9E', lw=1, ls=':')
    t_name = task.replace('monthly_', '').title()
    axes[0, i].set_title(f"{t_name} — Rewards"); axes[0, i].grid(True, alpha=0.3)
    axes[1, i].set_title(f"{t_name} — Energy"); axes[1, i].grid(True, alpha=0.3)
axes[0, 2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
fig.suptitle('Before vs After — Daily Trajectories', fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(f'{PLOTS_DIR}/training_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 7: Summary & Export

In [ ]:
# Cell 16: Final summary
print("=" * 67)
print("FINAL RESULTS")
print("=" * 67)
print(f"\n{'Task':<25s} {'Before':>10s} {'After':>10s} {'Delta':>10s} {'Smart':>10s}")
print("-" * 67)
for task in TASKS:
    b = before_results[task]["grader_score"]
    a = after_results[task]["grader_score"]
    s = baseline_results["smart"][task]["grader_score"]
    print(f"{task:<25s} {b:>10.4f} {a:>10.4f} {a-b:>+10.4f} {s:>10.4f}")

avg_b = np.mean([before_results[t]["grader_score"] for t in TASKS])
avg_a = np.mean([after_results[t]["grader_score"] for t in TASKS])
avg_s = np.mean([baseline_results["smart"][t]["grader_score"] for t in TASKS])
print("-" * 67)
print(f"{'AVERAGE':<25s} {avg_b:>10.4f} {avg_a:>10.4f} {avg_a-avg_b:>+10.4f} {avg_s:>10.4f}")

summary = {
    "model": MODEL_NAME,
    "training": "LoRA SFT (real weight updates)",
    "rounds": NUM_ROUNDS, "episodes_per_round": EPISODES_PER_ROUND,
    "before": {t: before_results[t]["grader_score"] for t in TASKS},
    "after": {t: after_results[t]["grader_score"] for t in TASKS},
    "smart_heuristic": {t: baseline_results["smart"][t]["grader_score"] for t in TASKS},
    "improvement": {t: after_results[t]["grader_score"] - before_results[t]["grader_score"] for t in TASKS},
    "training_log": training_log,
}
with open(f"{PLOTS_DIR}/training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

pd.DataFrame(training_log).to_csv(f"{PLOTS_DIR}/training_log.csv", index=False)

print(f"\nSaved to {PLOTS_DIR}/")
print("All results are from real LoRA weight updates on real environment runs.")

In [ ]:
# Cell 17: Save adapter
save_path = "./viraltest_trained_adapter"
peft_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"LoRA adapter saved to {save_path}")
print("Load with: PeftModel.from_pretrained(base_model, save_path)")